# いきものクイズを Google Colab で ひらく

最初のセルに「準備完了」が出るまで待ってから、起動セルを実行します。今回表示された新しいURLを開くと、4分野のゲームで遊べます。起動セルは遊んでいる間、実行中のままにしてください。

In [ ]:
# mainの最新版だけを取得します。履歴はダウンロードしません。
import os
from pathlib import Path
import shutil
import subprocess
import sys

SETUP_COMPLETE = False
REPO_URL = "https://github.com/no-kishimoto/kinoko-quiz.git"
project_dir = Path("/content/kinoko-quiz")
os.chdir("/content")
if project_dir.exists():
    shutil.rmtree(project_dir)

def run_with_progress(command, cwd):
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line.rstrip(), flush=True)
            code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
        if code:
            raise subprocess.CalledProcessError(code, command)

print("1/2：mainの最新版を取得しています。画像が多いため、完了までお待ちください。", flush=True)
run_with_progress(["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
                   "--progress", REPO_URL, str(project_dir)], "/content")
print("2/2：必要なライブラリをインストールしています。", flush=True)
run_with_progress([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], project_dir)
os.chdir(project_dir)
revision = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=project_dir, text=True).strip()
SETUP_COMPLETE = True
print(f"準備完了：main / {revision}。次の起動セルを実行してください。", flush=True)


In [ ]:
# 最初のセルで「準備完了」と表示されてから実行してください。
# このセルを動かしたまま、新しく表示された gradio.live のURLを開きます。
if not globals().get("SETUP_COMPLETE", False):
    raise RuntimeError("最初のセルが完了していません。最初のセルを実行し、準備完了を待ってください。")
subprocess.run([sys.executable, "app.py"], cwd=project_dir,
               env={**os.environ, "GRADIO_SHARE": "true"}, check=True)
